<h1>Sistemas de recomendación</h1></th></tr></tbody></table>



#### Este notebook contiene tanto la versión regularizada como la no regularizada del algoritmo de recomendacion de películas para realizar comparaciones entre ellos. El dataset con el que se trabaja tambien es “ex8_movies.mat” que contiene datos de películas valoradas por los usuarios del 1 al 5. Hay 943 usuarios que han valorado 1682 películas. Las películas presentan 10 características relativas a su contenido. 

In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.optimize as opt

#### 1. Cargamos y preparamos el dataset. Usaremos dos matrices: una matriz Y que contiene las distintas peliculas valoradas por un usuario, y R, una matriz binaria que indica si un usuario valoró o no una pelicula. 



In [2]:
#Carga de  datos 
print('Cargando dataset de puntaciones de películas.')
movies = sio.loadmat("ex8_movies.mat")
Y = movies['Y'] #[n_items, n_users] puntuaciones de 1-5
R = movies['R'] #[n_items, n_users] R(i,j)=1 si el usuario j puntuó pelicula i
print("Shape (forma) de Y: ", Y.shape)  #[n_items, features]
print("Shape (forma) de R: ", R.shape)  #[n_items, features]

print('\tPuntuación media para la primera película (Toy Story): ', Y[0, np.where(R[0, :] == 1)[0]].mean(), "/5\n")

#Cargar parámetros preentrenados (X, Theta, num_users, num_movies, num_features)
    
params_data = sio.loadmat('ex8_movieParams.mat')
X = params_data['X'] #Contiene las características de las películas.
Theta = params_data['Theta'] #Contiene las preferencias de los usuarios.

print("Shape (forma) de X: ", X.shape)  #[n_items, features]
print("Shape (forma) de Theta: ", Theta.shape)  #[features, n_users]


Cargando dataset de puntaciones de películas.
Shape (forma) de Y:  (1682, 943)
Shape (forma) de R:  (1682, 943)
	Puntuación media para la primera película (Toy Story):  3.8783185840707963 /5

Shape (forma) de X:  (1682, 10)
Shape (forma) de Theta:  (943, 10)


#### 2. Implementamos la función  de coste sin regularizar para un sistema de recomendación de filtrado colaborativo que llamaremos cofiCostFuncSinReg 



In [3]:
Y.shape
R.shape

(1682, 943)

In [4]:
def cofiCostFuncSinReg(params, Y, R, num_features):
    #Enrollamos la matrices X y theta a partir de params
    n_pelis = R.shape[0]
    n_user = R.shape[1]
    punto_corte = n_pelis * num_features
    X = params[:punto_corte]
    theta = params[punto_corte:]
    #Montamos matriz
    X = np.reshape(X, (n_pelis , num_features),'F')
    theta = np.reshape(theta,(num_features , n_user),'F')
    #Funcion de coste
    J = 0
    error = np.multiply((np.dot(X,theta) - Y),R)
    #multiplicamos por R ara solo tener las pelis vistas
    squared_error = np.power(error,2)
    J = (1/2) * np.sum(squared_error)
    return J

In [5]:
#Subconjunto de datos para agilizar la ejecución
users = 4
movies = 5
features = 3
X_sub = X[:movies, :features]
Theta_sub = Theta[:features, :users]
Y_sub = Y[:movies, :users]
R_sub = R[:movies, :users]
params = np.hstack((np.ravel(X_sub, order='F'), np.ravel(Theta_sub,order='F')))#AQUI ES DONDE APARECE POR PRIMERA VEZ PARAMS
J = cofiCostFuncSinReg(params, Y_sub, R_sub, features)
print("Coste sin regularización con los parámetros cargados ", J)


Coste sin regularización con los parámetros cargados  57.356479775998096


#### 3. Implementamos la función gradiente sin regularización cofiGradientFuncSinReg.



In [6]:
def cofiGradientFuncSinReg(params, Y, R, num_features):
     #Enrollamos la matrices X y theta a partir de params
    n_pelis = R.shape[0]
    n_user = R.shape[1]
    punto_corte = n_pelis * num_features
    X = params[:punto_corte]
    theta = params[punto_corte:]
    #Montamos matriz
    X = np.reshape(X, (n_pelis , num_features),'F')
    theta = np.reshape(theta,(num_features , n_user),'F')
    #Error
    error = np.multiply(np.dot(X,theta) - Y,R)
    #Inicializamos gradientes

    X_grad = np.zeros(X.shape)
    theta_grad = np.zeros(theta.shape)
    #calculograddientes
    X_grad = np.dot(error, theta.T)
    theta_grad = np.dot(X.T, error)
    #Aplanamos
    grad = np.hstack((np.ravel(X_grad,'F'), np.ravel(theta_grad, 'F')))
    return grad

In [7]:
grad = cofiGradientFuncSinReg(params, Y_sub, R_sub, features)
print("Gradiente sin regularización con los parámetros cargados: \n", grad)

Gradiente sin regularización con los parámetros cargados: 
 [  8.82173071  -0.912555    -1.15814916  -1.01861973  -0.80927332
   0.06240312  -1.61451267  -2.04902333  -1.80216476  -1.43178441
   5.28790198   1.38082478   1.75244347   1.54131572   1.22454499
 -15.05835243   8.17919126 -11.25116736  -6.45575963   2.46384767
  -7.35105818   0.           0.           0.           0.
   0.           0.        ]


#### 4. Implementamos ahora las versiones regularizadas del calculo del coste y del gradiente en funciones independientes.

In [8]:
def cofiCostFuncReg(params, Y, R, num_features, lambda_param):
    num_movies = Y.shape[0]
    num_users = Y.shape[1]
    X = np.reshape(params[:num_movies * num_features], (num_movies, num_features), 'F')
    Theta = np.reshape(params[num_movies * num_features:],(num_features, num_users), 'F')
    J = 0
    error = np.multiply(np.dot(X,Theta)- Y, R)
    error_cuadratico = (1/2) * np.power(error,2)
    J = (1/2)*np.sum(error_cuadratico)
    J = J + (lambda_param/2)*np.sum(np.power(X,2))
    J = J + (lambda_param/2)*np.sum(np.power(Theta,2))
    return J

In [9]:
def cofiGradientFuncReg(params, Y, R, num_features, lambda_param):
       #Enrollamos la matrices X y theta a partir de params
    n_pelis = R.shape[0]
    n_user = R.shape[1]
    punto_corte = n_pelis * num_features
    X = params[:punto_corte]
    theta = params[punto_corte:]
    #Montamos matriz
    X = np.reshape(X, (n_pelis , num_features),'F')
    theta = np.reshape(theta,(num_features , n_user),'F')
    #Error
    error = np.multiply(np.dot(X,theta) - Y,R)
    #Inicializamos gradientes

    X_grad = np.zeros(X.shape)
    theta_grad = np.zeros(theta.shape)
    #calculograddientes
    X_grad = np.dot(error, theta.T) + lambda_param * X
    theta_grad = np.dot(X.T, error) + lambda_param * theta
    #Aplanamos
    grad = np.hstack((np.ravel(X_grad,'F'), np.ravel(theta_grad, 'F')))
    return grad

In [10]:
# Evaluamos ambas funciones regularizadas
lambda_param = 1.5
J = cofiCostFuncReg(params, Y_sub, R_sub, features, lambda_param)
print("\n\nCoste con regularización con los parámetros cargados: ", J)

grad = cofiGradientFuncReg(params, Y_sub, R_sub, features, lambda_param)
print("Gradiente con regularización con los parámetros cargados: \n", grad)




Coste con regularización con los parámetros cargados:  37.3359363211612
Gradiente con regularización con los parámetros cargados: 
 [ 10.39475896   0.25872185  -0.19588587  -0.33819299   0.59703352
  -0.53794482  -2.19295154  -2.8708041   -3.00249242  -1.27264956
   7.07908115   2.16262146   1.62674891   2.56203765   1.76747442
 -14.630187     8.93671108 -11.89904221  -8.98215726   1.78187498
  -8.06926491   0.39440815   0.47619366   1.27006666  -0.43097597
  -0.17263041  -0.01759684]


#### 5. Inicializamos de forma aleatoria con valores pequeños las matrices X y Theta. Entrenamos con fmin



In [11]:
movies = Y.shape[0]  # 1682
users = Y.shape[1]  # 943
features = 10
lambda_param = 1.5
maxiter = 200

# Inicialización de X y Theta
X = np.random.rand(movies, features) * (2*0.12)
Theta = np.random.rand(features, users) * (2*0.12)
params = np.hstack((np.ravel(X,order='F'), np.ravel(Theta,order='F'))) # Desenrollar: primero X luego Theta
# Algoritmo de optimización
fmin_1 = opt.fmin_cg(maxiter=maxiter, f=cofiCostFuncReg, x0=params, fprime=cofiGradientFuncReg,
                  args=(Y, R, features, lambda_param))
# Enrollar el resultado
X_fmin = np.reshape(fmin_1[:movies*features],(movies,features),'F')
Theta_fmin = np.reshape(fmin_1[movies*features:], (features,users), 'F')


         Current function value: 21242.411222
         Iterations: 74
         Function evaluations: 233
         Gradient evaluations: 221


C:\Users\varea.LAPTOP-P3D215RM\anaconda3\envs\entornoIA2425\lib\site-packages\scipy\optimize\_optimize.py:1659: OptimizeWarning: Desired error not necessarily achieved due to precision loss.
  res = _minimize_cg(f, x0, args, fprime, callback=callback, c1=c1, c2=c2,


#### 6. Realizamos las predicciones de las 10 películas con mejores puntuaciones para un usuario en concreto



In [12]:
predictions = np.dot(X_fmin,Theta_fmin)
#Solo el usuario j
j = 200
res_user = np.zeros((movies, 1))
pred_userj = predictions[:,j] # Seleccionar el usuario j

#Para cada película: A las que tenían valor previo le ponemos un 0 y a las que hemos predicho el valor de su predicción
for i in range(movies):
    res_user[i,0] = np.where(R[i,j]==0, predictions[i,j],0)
idx = np.argsort(res_user, axis=0)[::-1] # Ordenar por las predicciones de menor a mayor y coger sus índice. [::-1] significa que le damos la vuelta a la salida: es decir lo colocamos de mayor a menor

#Leemos el fichero con los nombres de cada película
movie_idx = {}
f = open('movie_ids.txt',encoding = 'ISO-8859-1')
for line in f:
    tokens = line.split(' ')
    tokens[-1] = tokens[-1][:-1]
    movie_idx[int(tokens[0]) - 1] = ' '.join(tokens[1:])
print("Predicciones de puntuación para las 10 mejores películas:")
for i in range(10):
    j = int(idx[i])
    print('Puntuación predicha de {0} para la película {1}.'.format(str(float(res_user[j])), movie_idx[j]))


Predicciones de puntuación para las 10 mejores películas:
Puntuación predicha de 4.777477925029082 para la película Nightmare Before Christmas, The (1993).
Puntuación predicha de 4.746667575378876 para la película Young Frankenstein (1974).
Puntuación predicha de 4.70999997422157 para la película Willy Wonka and the Chocolate Factory (1971).
Puntuación predicha de 4.652557988843826 para la película Last Supper, The (1995).
Puntuación predicha de 4.6351049455241915 para la película Paradise Lost: The Child Murders at Robin Hood Hills (1996).
Puntuación predicha de 4.521724272205164 para la película Mighty Aphrodite (1995).
Puntuación predicha de 4.432773549193044 para la película Wings of Desire (1987).
Puntuación predicha de 4.395433389026299 para la película Sleeper (1973).
Puntuación predicha de 4.377279217503448 para la película Face/Off (1997).
Puntuación predicha de 4.355730892588246 para la película Monty Python and the Holy Grail (1974).


C:\Users\varea.LAPTOP-P3D215RM\AppData\Local\Temp\ipykernel_16860\506104746.py:21: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  j = int(idx[i])
C:\Users\varea.LAPTOP-P3D215RM\AppData\Local\Temp\ipykernel_16860\506104746.py:22: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('Puntuación predicha de {0} para la película {1}.'.format(str(float(res_user[j])), movie_idx[j]))
